# Checking the optimisations

Checks that each function in `optimisations.py` does what its docstring says, then measures how much each one lowers the cost.

1. **Correctness checks**: each function is run many times on benchmark instances (and on tiny random instances, where the answer can be found by brute force). Every run is checked against rules that must always hold, e.g. no customer is lost, routes stay within capacity, the cost never goes up. The summary table lists every check with how many runs passed and failed, and an example of the first failure.
2. **How much each optimisation helps**:
   - `two_opt` vs `optimised_two_opt` on routes of different lengths
   - each step of RFCS (giant tour → 2-opt → split → relocate/swap → 2-opt → ruin and recreate)
   - Clarke-Wright on its own vs with the suggested 2-opt and relocate/swap added afterwards (applied here in the notebook; `clark_wright_savings.py` is not changed)

Nothing here needs a time limit, so it runs in this process (no `algorithm_runner.py`).

In [17]:
import sys
sys.path.append('../')

import copy
import random
import time

import pandas as pd
pd.set_option('display.max_rows', 200)     # show whole tables

import optimisations as opt
import route_first_cluster_second as rfcs
from clark_wright_savings import run_clarke_wright
from final_solution import build_nearest_neighbours, pack_by_demand, starting_overload_penalty
from run_benchmarks import find_instances, read_instance, read_solution
from utilities import calculate_cost, is_feasible

## Settings

In [18]:
SETS = ['A', 'B', 'P', 'X']     # benchmark sets to use
SAMPLES_PER_SET = 4             # smallest instances per set for the correctness checks
TRIALS = 20                     # random runs per function and instance
SMALL_TRIALS = 300              # tiny random instances checked against brute force
CW_SAMPLES_PER_SET = 5          # instances per set for the Clarke-Wright comparison (None = all)
RUIN_RECREATE_ITERATIONS = 200  # ruin and recreate iterations per instance in the RFCS step comparison
SEED = 0                        # so every run of this notebook gives the same results

## Instances and helpers

In [19]:
def load_instance(path):
    '''Reads an instance and everything the optimisations need, built the same way as final_solution.find_solution.'''
    cust_demands, dist_matrix, vehicle_cap, num_vehicles = read_instance(path)
    opt_path = path.with_name(path.stem + '_opt.txt')
    optimal_cost, optimal_routes = read_solution(opt_path) if opt_path.exists() else (None, None)
    return {
        'name': path.stem, 'set': path.parent.name,
        'cust_demands': cust_demands, 'dist_matrix': dist_matrix,
        'vehicle_cap': vehicle_cap, 'num_vehicles': num_vehicles,
        'nearest_neighbours': build_nearest_neighbours(dist_matrix),
        'penalty': starting_overload_penalty(cust_demands, dist_matrix),
        'optimal_cost': optimal_cost,
        # some X instances' optimal solutions use more routes than num_vehicles, so they cannot be compared against
        'opt_feasible': optimal_routes is not None and is_feasible(optimal_routes, cust_demands, vehicle_cap, num_vehicles),
    }


instances = [load_instance(path) for s in SETS for path in find_instances([s], None)[:SAMPLES_PER_SET]]
pd.DataFrame([{key: inst[key] for key in ('set', 'name', 'vehicle_cap', 'num_vehicles', 'optimal_cost', 'opt_feasible')}
              for inst in instances])

,set,name,vehicle_cap,num_vehicles,optimal_cost,opt_feasible
0,A,A-n32-k5,100,5,784,True
1,A,A-n33-k5,100,5,661,True
2,A,A-n33-k6,100,6,742,True
3,A,A-n34-k5,100,5,778,True
4,B,B-n31-k5,100,5,672,True
5,B,B-n34-k5,100,5,788,True
6,B,B-n35-k5,100,5,955,True
7,B,B-n38-k6,100,6,805,True
8,P,P-n16-k8,35,8,450,True
9,P,P-n19-k2,160,2,212,True


In [20]:
# every check is recorded here; the summary is at the end of the correctness checks
check_rows = []

def check(function, description, passed, example=''):
    '''Records one check. For a failure, example says how to reproduce it.'''
    check_rows.append({'function': function, 'check': description, 'passed': bool(passed),
                       'example': '' if passed else example})


def customers_of(routes):
    '''Every customer in the routes, sorted (a customer visited twice appears twice).'''
    return sorted(customer for route in routes for customer in route[1:-1])


def all_customers(inst):
    return list(range(1, len(inst['cust_demands'])))


def loads_of(routes, cust_demands):
    return [sum(cust_demands[customer] for customer in route) for route in routes]


def total_overload(routes, cust_demands, vehicle_cap):
    return sum(max(0, load - vehicle_cap) for load in loads_of(routes, cust_demands))


def depot_at_both_ends(routes):
    return all(route[0] == 0 and route[-1] == 0 and 0 not in route[1:-1] for route in routes)


def random_tour(inst, rng):
    tour = all_customers(inst)
    rng.shuffle(tour)
    return tour


def capacity_split(tour, cust_demands, vehicle_cap):
    '''Cuts the tour whenever the next customer would not fit: within capacity, but may use more than num_vehicles routes.'''
    routes, route, load = [], [0], 0
    for customer in tour:
        if load + cust_demands[customer] > vehicle_cap:
            routes.append(route + [0])
            route, load = [0], 0
        route.append(customer)
        load += cust_demands[customer]
    routes.append(route + [0])
    return routes


def feasible_start(inst, rng, attempts=20):
    '''A feasible solution to start ruin and recreate from (split of an RFCS giant tour), or None if none was found.'''
    for _ in range(attempts):
        tour = rfcs.build_giant_tour(inst['dist_matrix'], inst['nearest_neighbours'], rng)
        routes = opt.split_within_vehicle_capacity(tour, inst['cust_demands'], inst['dist_matrix'],
                                                   inst['vehicle_cap'], inst['num_vehicles'])
        if routes is not None:
            return routes
    return None


def has_improving_two_opt_move(path, dist_matrix):
    '''Brute force: True if reversing any segment (ends fixed) would make the path shorter by more than 1e-9.'''
    for i in range(len(path) - 3):
        for j in range(i + 2, len(path) - 1):
            saving = (dist_matrix[path[i]][path[i + 1]] + dist_matrix[path[j]][path[j + 1]]
                      - dist_matrix[path[i]][path[j]] - dist_matrix[path[i + 1]][path[j + 1]])
            if saving > 1e-9:
                return True
    return False


def path_length(path, dist_matrix):
    return sum(dist_matrix[a][b] for a, b in zip(path, path[1:]))


rng = random.Random(SEED)

## 1. Correctness checks

### `two_opt` and `optimised_two_opt`
Both must keep the same stops with the ends fixed, and never make the path longer. `two_opt` should also leave no improving reversal (a true 2-opt optimum). `optimised_two_opt` only tries near neighbours, so it is *not* expected to reach a 2-opt optimum; how often it does is recorded in section 2.

In [21]:
for inst in instances:
    dist_matrix = inst['dist_matrix']
    customers = all_customers(inst)
    for trial in range(TRIALS):
        size = rng.randint(2, min(60, len(customers)))
        route = [0, *rng.sample(customers, size), 0]
        example = f"{inst['name']} route={route}"

        for name, run in (('two_opt', lambda path: opt.two_opt(path, dist_matrix)),
                          ('optimised_two_opt', lambda path: opt.optimised_two_opt(path, dist_matrix, inst['nearest_neighbours']))):
            path = list(route)
            result = run(path)
            check(name, 'changes the path in place and returns it', result is path, example)
            check(name, 'keeps the same stops', sorted(result) == sorted(route), example)
            check(name, 'keeps the first and last stop fixed', result[0] == route[0] and result[-1] == route[-1], example)
            check(name, 'never makes the path longer', path_length(result, dist_matrix) <= path_length(route, dist_matrix) + 1e-9, example)
            if name == 'two_opt':
                check(name, 'leaves no improving reversal (2-opt optimum)', not has_improving_two_opt_move(result, dist_matrix), example)

    # giant tour (no depot), as build_giant_tour uses it
    for trial in range(TRIALS // 4):
        tour = random_tour(inst, rng)
        result = opt.optimised_two_opt(list(tour), dist_matrix, inst['nearest_neighbours'])
        example = f"{inst['name']} giant tour, trial {trial}"
        check('optimised_two_opt', 'giant tour: keeps the same customers', sorted(result) == sorted(tour), example)
        check('optimised_two_opt', 'giant tour: never makes it longer', path_length(result, dist_matrix) <= path_length(tour, dist_matrix) + 1e-9, example)

### `relocate_and_swap`
Without a penalty: every customer kept exactly once, every route within capacity, cost never higher, no new routes, and the input routes left unchanged. With a penalty: the penalised cost (distance + penalty × overload) never higher.

In [22]:
for inst in instances:
    cust_demands, dist_matrix, vehicle_cap = inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap']
    for trial in range(TRIALS):
        example = f"{inst['name']} trial {trial}"

        # without a penalty: start from routes within capacity
        routes = capacity_split(random_tour(inst, rng), cust_demands, vehicle_cap)
        before = copy.deepcopy(routes)
        result = opt.relocate_and_swap(routes, cust_demands, dist_matrix, vehicle_cap, inst['nearest_neighbours'], rng)
        check('relocate_and_swap', 'no penalty: input routes unchanged', routes == before, example)
        check('relocate_and_swap', 'no penalty: every customer exactly once', customers_of(result) == all_customers(inst), example)
        check('relocate_and_swap', 'no penalty: depot at both ends, no empty routes', depot_at_both_ends(result) and all(len(r) > 2 for r in result), example)
        check('relocate_and_swap', 'no penalty: every route within capacity', total_overload(result, cust_demands, vehicle_cap) == 0, example)
        check('relocate_and_swap', 'no penalty: no new routes', len(result) <= len(routes), example)
        check('relocate_and_swap', 'no penalty: cost never higher', calculate_cost(dist_matrix, result) <= calculate_cost(dist_matrix, routes) + 1e-6, example)

        # with a penalty: start from overloaded routes (split allowing overload)
        routes = opt.split_with_vehicle_overload(random_tour(inst, rng), cust_demands, dist_matrix, vehicle_cap,
                                                 inst['num_vehicles'], inst['penalty'])
        if routes is None:
            continue
        penalised = lambda rs: calculate_cost(dist_matrix, rs) + inst['penalty'] * total_overload(rs, cust_demands, vehicle_cap)
        result = opt.relocate_and_swap(routes, cust_demands, dist_matrix, vehicle_cap, inst['nearest_neighbours'], rng, inst['penalty'])
        check('relocate_and_swap', 'penalty: every customer exactly once', customers_of(result) == all_customers(inst), example)
        check('relocate_and_swap', 'penalty: distance + penalty x overload never higher', penalised(result) <= penalised(routes) + 1e-6, example)

### `split_within_vehicle_capacity` and `split_with_vehicle_overload`
On benchmark instances: the routes must keep the tour order, use at most `num_vehicles` routes, and stay within capacity (or 1.5 × capacity with overload).

On tiny random instances (up to 10 customers), every possible way of cutting the tour is tried, so the cost of the split must equal the brute-force best, and it must return `None` exactly when no cut exists.

In [23]:
def brute_force_split(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles, penalty=None):
    '''Tries every way to cut the tour. penalty=None: within capacity; otherwise up to MAX_OVERLOAD_FACTOR x capacity, with the penalty.
    Returns the lowest cost (infinity if there is no cut).'''
    best = float('inf')
    max_load = vehicle_cap if penalty is None else opt.MAX_OVERLOAD_FACTOR * vehicle_cap
    for cuts in range(1 << (len(tour) - 1)):
        routes, start = [], 0
        for position in range(1, len(tour)):
            if cuts >> (position - 1) & 1:
                routes.append([0, *tour[start:position], 0])
                start = position
        routes.append([0, *tour[start:], 0])
        loads = loads_of(routes, cust_demands)
        if len(routes) > num_vehicles or max(loads) > max_load:
            continue
        cost = calculate_cost(dist_matrix, routes)
        if penalty is not None:
            cost += penalty * sum(max(0, load - vehicle_cap) for load in loads)
        best = min(best, cost)
    return best


def tiny_instance(rng):
    '''A random instance with 3 to 10 customers on a 100 x 100 grid.'''
    num_customers = rng.randint(3, 10)
    points = [(rng.uniform(0, 100), rng.uniform(0, 100)) for _ in range(num_customers + 1)]
    dist_matrix = [[((xa - xb) ** 2 + (ya - yb) ** 2) ** 0.5 for xb, yb in points] for xa, ya in points]
    cust_demands = [0] + [rng.randint(1, 10) for _ in range(num_customers)]
    vehicle_cap = rng.randint(10, 30)
    num_vehicles = rng.randint(1, num_customers)
    return cust_demands, dist_matrix, vehicle_cap, num_vehicles


def split_cost(routes, cust_demands, dist_matrix, vehicle_cap, penalty=None):
    if routes is None:
        return float('inf')
    cost = calculate_cost(dist_matrix, routes)
    if penalty is not None:
        cost += penalty * total_overload(routes, cust_demands, vehicle_cap)
    return cost


# tiny instances: compare with brute force
for trial in range(SMALL_TRIALS):
    cust_demands, dist_matrix, vehicle_cap, num_vehicles = tiny_instance(rng)
    tour = list(range(1, len(cust_demands)))
    rng.shuffle(tour)
    example = f'tiny instance trial {trial}: demands={cust_demands}, cap={vehicle_cap}, vehicles={num_vehicles}, tour={tour}'

    routes = opt.split_within_vehicle_capacity(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles)
    best = brute_force_split(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles)
    check('split_within_vehicle_capacity', 'tiny: returns None exactly when no cut fits', (routes is None) == (best == float('inf')), example)
    check('split_within_vehicle_capacity', 'tiny: cost equals the brute-force best',
          abs(split_cost(routes, cust_demands, dist_matrix, vehicle_cap) - best) < 1e-6 or best == float('inf'), example)

    penalty = rng.uniform(0.5, 20)
    routes = opt.split_with_vehicle_overload(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles, penalty)
    best = brute_force_split(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles, penalty)
    check('split_with_vehicle_overload', 'tiny: returns None exactly when no cut fits', (routes is None) == (best == float('inf')), example)
    check('split_with_vehicle_overload', 'tiny: distance + penalty x overload equals the brute-force best',
          abs(split_cost(routes, cust_demands, dist_matrix, vehicle_cap, penalty) - best) < 1e-6 or best == float('inf'), example)

# benchmark instances: the rules every split must follow
for inst in instances:
    cust_demands, dist_matrix, vehicle_cap, num_vehicles = inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap'], inst['num_vehicles']
    for trial in range(TRIALS):
        tour = rfcs.build_giant_tour(dist_matrix, inst['nearest_neighbours'], rng)
        example = f"{inst['name']} trial {trial}"
        for name, routes, max_load in (
                ('split_within_vehicle_capacity',
                 opt.split_within_vehicle_capacity(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles), vehicle_cap),
                ('split_with_vehicle_overload',
                 opt.split_with_vehicle_overload(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles, inst['penalty']),
                 opt.MAX_OVERLOAD_FACTOR * vehicle_cap)):
            if routes is None:
                continue
            check(name, 'keeps the tour order', [c for route in routes for c in route[1:-1]] == tour, example)
            check(name, 'depot at both ends, no empty routes', depot_at_both_ends(routes) and all(len(r) > 2 for r in routes), example)
            check(name, 'at most num_vehicles routes', len(routes) <= num_vehicles, example)
            check(name, 'every route within its load limit', max(loads_of(routes, cust_demands)) <= max_load, example)

### `vehicle_reduction`, `cheapest_insertion` and `repair_overloads`
Checked in the order Clarke-Wright uses them: too many routes → remove the lightest → insert their customers → repair.

In [24]:
for inst in instances:
    cust_demands, dist_matrix, vehicle_cap, num_vehicles = inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap'], inst['num_vehicles']
    for trial in range(TRIALS):
        example = f"{inst['name']} trial {trial}"
        routes = capacity_split(random_tour(inst, rng), cust_demands, vehicle_cap)     # usually more than num_vehicles routes
        before = copy.deepcopy(routes)

        kept, removed = opt.vehicle_reduction(routes, cust_demands, num_vehicles)
        check('vehicle_reduction', 'input routes unchanged', routes == before, example)
        check('vehicle_reduction', 'keeps min(num_vehicles, routes) routes', len(kept) == min(num_vehicles, len(routes)), example)
        check('vehicle_reduction', 'kept + removed customers = every customer once', sorted(customers_of(kept) + removed) == all_customers(inst), example)
        if removed:
            removed_loads = [load for route, load in zip(routes, loads_of(routes, cust_demands)) if route not in kept]
            check('vehicle_reduction', 'removes the lightest routes', max(removed_loads) <= min(loads_of(kept, cust_demands)), example)

        inserted = opt.cheapest_insertion(kept, removed, cust_demands, dist_matrix, vehicle_cap)
        check('cheapest_insertion', 'every customer exactly once', customers_of(inserted) == all_customers(inst), example)
        check('cheapest_insertion', 'no new routes, depot at both ends', len(inserted) == len(kept) and depot_at_both_ends(inserted), example)

        repaired = opt.repair_overloads(inserted, cust_demands, dist_matrix, vehicle_cap, inst['nearest_neighbours'],
                                        rng, inst['penalty'], opt.MAX_PENALTY_DOUBLINGS)
        if repaired is not None:
            check('repair_overloads', 'result: every customer exactly once', customers_of(repaired) == all_customers(inst), example)
            check('repair_overloads', 'result: every route within capacity', total_overload(repaired, cust_demands, vehicle_cap) == 0, example)
            check('repair_overloads', 'result: no new routes', len(repaired) <= len(inserted), example)
            check('repair_overloads', 'result is feasible', is_feasible(repaired, cust_demands, vehicle_cap, num_vehicles), example)

### `ruin`, `recreate` and `ruin_and_recreate`
Started from a feasible solution, as `run_rfcs` does.

In [25]:
for inst in instances:
    cust_demands, dist_matrix, vehicle_cap, num_vehicles = inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap'], inst['num_vehicles']
    start_routes = feasible_start(inst, rng)
    if start_routes is None:
        print(f"{inst['name']}: no feasible starting solution found, ruin and recreate not checked")
        continue

    num_customers = len(cust_demands) - 1
    max_remove = max(1, min(opt.MAX_REMOVE, num_customers // 4))
    min_remove = min(opt.MIN_REMOVE, max_remove)
    for trial in range(TRIALS):
        num_remove = rng.randint(min_remove, max_remove)
        example = f"{inst['name']} trial {trial}, num_remove={num_remove}"
        before = copy.deepcopy(start_routes)

        ruined, removed = opt.ruin(start_routes, inst['nearest_neighbours'], num_remove, rng)
        check('ruin', 'input routes unchanged', start_routes == before, example)
        check('ruin', 'removes exactly num_remove different customers', len(removed) == len(set(removed)) == num_remove, example)
        check('ruin', 'removed customers are not in the routes', set(removed).isdisjoint(customers_of(ruined)), example)
        check('ruin', 'kept + removed customers = every customer once', sorted(customers_of(ruined) + removed) == all_customers(inst), example)
        check('ruin', 'depot at both ends, no empty routes', depot_at_both_ends(ruined) and all(len(r) > 2 for r in ruined), example)

        recreated = opt.recreate(copy.deepcopy(ruined), removed, cust_demands, dist_matrix, vehicle_cap, num_vehicles,
                                 inst['nearest_neighbours'], rng)
        if recreated is not None:
            check('recreate', 'result is feasible', is_feasible(recreated, cust_demands, vehicle_cap, num_vehicles), example)

        routes, removed = opt.ruin_and_recreate(start_routes, cust_demands, dist_matrix, vehicle_cap, num_vehicles,
                                                inst['nearest_neighbours'], num_remove, rng)
        check('ruin_and_recreate', 'input routes unchanged', start_routes == before, example)
        if routes is not None:
            check('ruin_and_recreate', 'result is feasible', is_feasible(routes, cust_demands, vehicle_cap, num_vehicles), example)

P-n16-k8: no feasible starting solution found, ruin and recreate not checked
X-n101-k25: no feasible starting solution found, ruin and recreate not checked


### Summary of the correctness checks

In [26]:
checks = pd.DataFrame(check_rows)
summary = (checks.groupby(['function', 'check'], sort=False)
                 .agg(runs=('passed', 'size'), failed=('passed', lambda passed: (~passed).sum()),
                      first_failure=('example', lambda examples: next((e for e in examples if e), ''))))
print(f"{len(checks)} checks, {(~checks['passed']).sum()} failed")
summary

13813 checks, 0 failed


runs  \
function                      check                                                      
two_opt                       changes the path in place and returns it             320   
                              keeps the same stops                                 320   
                              keeps the first and last stop fixed                  320   
                              never makes the path longer                          320   
                              leaves no improving reversal (2-opt optimum)         320   
optimised_two_opt             changes the path in place and returns it             320   
                              keeps the same stops                                 320   
                              keeps the first and last stop fixed                  320   
                              never makes the path longer                          320   
                              giant tour: keeps the same customers                  80   
                              giant tour: never makes it longer                     80   
relocate_and_swap             no penalty: input routes unchanged                   320   
                              no penalty: every customer exactly once              320   
                              no penalty: depot at both ends, no empty routes      320   
                              no penalty: every route within capacity              320   
                              no penalty: no new routes                            320   
                              no penalty: cost never higher                        320   
                              penalty: every customer exactly once                 320   
                              penalty: distance + penalty x overload never hi...   320   
split_within_vehicle_capacity tiny: returns None exactly when no cut fits          300   
                              tiny: cost equals the brute-force best               300   
split_with_vehicle_overload   tiny: returns None exactly when no cut fits          300   
                              tiny: distance + penalty x overload equals the ...   300   
split_within_vehicle_capacity keeps the tour order                                 174   
                              depot at both ends, no empty routes                  174   
                              at most num_vehicles routes                          174   
                              every route within its load limit                    174   
split_with_vehicle_overload   keeps the tour order                                 320   
                              depot at both ends, no empty routes                  320   
                              at most num_vehicles routes                          320   
                              every route within its load limit                    320   
vehicle_reduction             input routes unchanged                               320   
                              keeps min(num_vehicles, routes) routes               320   
                              kept + removed customers = every customer once       320   
cheapest_insertion            every customer exactly once                          320   
                              no new routes, depot at both ends                    320   
repair_overloads              result: every customer exactly once                  274   
                              result: every route within capacity                  274   
                              result: no new routes                                274   
                              result is feasible                                   274   
vehicle_reduction             removes the lightest routes                          123   
ruin                          input routes unchanged                               280   
                              removes exactly num_remove different customers       280   
                              re

In [27]:
# only the checks that failed at least once
summary[summary['failed'] > 0]

,,runs,failed,first_failure
function,check,,,


## 2. How much each optimisation helps

### `two_opt` vs `optimised_two_opt`
On random routes of each length: how much shorter each makes the route, how long it takes, and how often `optimised_two_opt` reaches a true 2-opt optimum.

In [28]:
largest = max(instances, key=lambda inst: len(inst['cust_demands']))      # needs enough customers for the longest routes
dist_matrix, nearest_neighbours = largest['dist_matrix'], largest['nearest_neighbours']
rows = []
for size in (5, 10, 20, 40, 80):
    if size > len(largest['cust_demands']) - 1:
        continue
    for trial in range(TRIALS):
        route = [0, *rng.sample(all_customers(largest), size), 0]
        start_length = path_length(route, dist_matrix)
        for name, run in (('two_opt', lambda path: opt.two_opt(path, dist_matrix)),
                          ('optimised_two_opt', lambda path: opt.optimised_two_opt(path, dist_matrix, nearest_neighbours))):
            path = list(route)
            time_start = time.perf_counter()
            run(path)
            elapsed = time.perf_counter() - time_start
            rows.append({'route_customers': size, 'function': name,
                         'shorter_pct': 100 * (start_length - path_length(path, dist_matrix)) / start_length,
                         'time_ms': 1000 * elapsed,
                         'reaches_2opt_optimum': not has_improving_two_opt_move(path, dist_matrix)})

print(f"routes from {largest['name']}")
pd.DataFrame(rows).groupby(['route_customers', 'function'])[['shorter_pct', 'time_ms', 'reaches_2opt_optimum']].mean().round(3)

routes from X-n115-k10


shorter_pct  time_ms  reaches_2opt_optimum
route_customers function                                                     
5               optimised_two_opt       15.548    0.006                  0.20
                two_opt                 23.017    0.004                  1.00
10              optimised_two_opt       38.041    0.015                  0.05
                two_opt                 46.230    0.010                  1.00
20              optimised_two_opt       57.476    0.057                  0.05
                two_opt                 59.575    0.039                  1.00
40              optimised_two_opt       71.398    0.179                  0.05
                two_opt                 71.748    0.173                  1.00
80              optimised_two_opt       80.560    0.546                  0.05
                two_opt                 80.889    0.768                  1.00

### RFCS: what each step does
For random giant tours, the cost after each step of a restart in `run_rfcs`, then `RUIN_RECREATE_ITERATIONS` iterations of ruin and recreate (accepting only improvements, as `run_rfcs` does). `lowered_pct` is the share of runs where that step lowered the cost.

In [29]:
step_rows = []
for inst in instances:
    cust_demands, dist_matrix, vehicle_cap, num_vehicles = inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap'], inst['num_vehicles']
    nearest_neighbours = inst['nearest_neighbours']
    for trial in range(3):
        def record(step, cost, before):
            step_rows.append({'instance': inst['name'], 'step': step, 'cost': cost,
                              'lowered': cost < before - 1e-9, 'lower_by_pct': 100 * (before - cost) / before})

        # giant tour: cost as a single round trip from the depot
        tour = rfcs.randomised_nn_tour(dist_matrix, nearest_neighbours, rng)
        tour_cost = calculate_cost(dist_matrix, [[0, *tour, 0]])
        opt.optimised_two_opt(tour, dist_matrix, nearest_neighbours)
        record('1. optimised_two_opt on the giant tour', calculate_cost(dist_matrix, [[0, *tour, 0]]), tour_cost)

        routes, method = rfcs.partition_giant_tour(tour, cust_demands, dist_matrix, vehicle_cap, num_vehicles,
                                                   nearest_neighbours, rng, inst['penalty'])
        if routes is None or not is_feasible(routes, cust_demands, vehicle_cap, num_vehicles):
            continue
        cost = calculate_cost(dist_matrix, routes)

        routes = rfcs.improve_between_routes(routes, cust_demands, dist_matrix, vehicle_cap, nearest_neighbours, rng)
        new_cost = calculate_cost(dist_matrix, routes)
        record('2. relocate_and_swap after the split', new_cost, cost)
        cost = new_cost

        routes = rfcs.improve_within_routes(routes, dist_matrix)
        new_cost = calculate_cost(dist_matrix, routes)
        record('3. two_opt on each route', new_cost, cost)
        cost = new_cost

        # ruin and recreate from this solution, keeping only improvements
        start_cost = cost
        num_customers = len(cust_demands) - 1
        max_remove = max(1, min(opt.MAX_REMOVE, num_customers // 4))
        min_remove = min(opt.MIN_REMOVE, max_remove)
        for _ in range(RUIN_RECREATE_ITERATIONS):
            new_routes, removed = opt.ruin_and_recreate(routes, cust_demands, dist_matrix, vehicle_cap, num_vehicles,
                                                        nearest_neighbours, rng.randint(min_remove, max_remove), rng)
            if new_routes is None:
                continue
            new_routes = rfcs.improve_between_routes(new_routes, cust_demands, dist_matrix, vehicle_cap, nearest_neighbours,
                                                     rng, customers_to_check=removed)
            new_routes = rfcs.improve_within_routes(new_routes, dist_matrix, customers_to_check=removed)
            new_cost = calculate_cost(dist_matrix, new_routes)
            if new_cost < cost - 1e-9:
                routes, cost = new_routes, new_cost
        record(f'4. {RUIN_RECREATE_ITERATIONS} ruin and recreate iterations', cost, start_cost)

steps = pd.DataFrame(step_rows)
(steps.groupby('step').agg(runs=('cost', 'size'), lowered_pct=('lowered', lambda lowered: 100 * lowered.mean()),
                           mean_lower_by_pct=('lower_by_pct', 'mean')).round(2))

,runs,lowered_pct,mean_lower_by_pct
step,,,
1. optimised_two_opt on the giant tour,48,100.00,34.05
2. relocate_and_swap after the split,39,53.85,1.61
3. two_opt on each route,39,94.87,3.35
4. 200 ruin and recreate iterations,39,89.74,3.52


### Clarke-Wright with the suggested optimisations
Runs `run_clarke_wright` as it is now, then applies each suggested optimisation to its result here in the notebook. Gaps are only shown for instances whose optimal solution is feasible (`opt_feasible`).

In [30]:
def clarke_wright_routes(inst):
    '''The routes run_clarke_wright saves, or None if it found no feasible solution.'''
    state = {'best': {'solution': (float('inf'), None)}, 'best_cost': float('inf')}
    run_clarke_wright(inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap'], inst['num_vehicles'],
                      inst['nearest_neighbours'], inst['penalty'], random.Random(SEED), state)
    return state['best']['solution'][1]


def relocate_swap_then_two_opt(routes, inst):
    routes = opt.relocate_and_swap(routes, inst['cust_demands'], inst['dist_matrix'], inst['vehicle_cap'],
                                   inst['nearest_neighbours'], random.Random(SEED))
    return [opt.two_opt(route, inst['dist_matrix']) for route in routes]


CW_VARIANTS = {
    'clarke_wright': lambda routes, inst: routes,
    '+ two_opt': lambda routes, inst: [opt.two_opt(list(route), inst['dist_matrix']) for route in routes],
    '+ optimised_two_opt': lambda routes, inst: [opt.optimised_two_opt(list(route), inst['dist_matrix'], inst['nearest_neighbours'])
                                                 for route in routes],
    '+ relocate_and_swap + two_opt': relocate_swap_then_two_opt,
}

cw_rows = []
for s in SETS:
    paths = find_instances([s], None)
    for path in paths[:CW_SAMPLES_PER_SET] if CW_SAMPLES_PER_SET else paths:
        inst = load_instance(path)
        base_routes = clarke_wright_routes(inst)
        for variant, improve in CW_VARIANTS.items():
            row = {'set': s, 'instance': inst['name'], 'variant': variant, 'opt_feasible': inst['opt_feasible']}
            if base_routes is not None:
                time_start = time.perf_counter()
                routes = improve(copy.deepcopy(base_routes), inst)
                row['time_ms'] = 1000 * (time.perf_counter() - time_start)
                row['feasible'] = is_feasible(routes, inst['cust_demands'], inst['vehicle_cap'], inst['num_vehicles'])
                row['cost'] = calculate_cost(inst['dist_matrix'], routes)
                if inst['opt_feasible'] and row['feasible']:
                    row['gap_pct'] = 100 * (row['cost'] - inst['optimal_cost']) / inst['optimal_cost']
            else:
                row['feasible'] = False     # Clarke-Wright found no feasible solution, so there is nothing to improve
            cw_rows.append(row)

cw = pd.DataFrame(cw_rows)
cw.groupby(['set', 'variant'], sort=False)[['gap_pct', 'time_ms']].mean().round(3)

gap_pct  time_ms
set variant                                        
A   clarke_wright                    5.064    0.009
    + two_opt                        4.302    0.020
    + optimised_two_opt              4.302    0.048
    + relocate_and_swap + two_opt    4.035    0.300
B   clarke_wright                    2.088    0.008
    + two_opt                        1.759    0.020
    + optimised_two_opt              1.759    0.050
    + relocate_and_swap + two_opt    1.596    0.345
P   clarke_wright                    7.994    0.005
    + two_opt                        7.994    0.010
    + optimised_two_opt              7.994    0.021
    + relocate_and_swap + two_opt    7.994    0.060
X   clarke_wright                    6.539    0.021
    + two_opt                        6.088    0.093
    + optimised_two_opt              6.091    0.209
    + relocate_and_swap + two_opt    5.812    1.073

In [31]:
# gap from optimal (%) for each instance and variant ('infeasible' = Clarke-Wright found no feasible solution;
# 'opt infeasible' = the optimal solution uses too many vehicles, so there is nothing to compare with)
gaps = cw.pivot(index=['set', 'instance'], columns='variant', values='gap_pct')[list(CW_VARIANTS)].round(2).astype(object)
for (s, name), group in cw.groupby(['set', 'instance'], sort=False):
    if not group['opt_feasible'].iloc[0]:
        gaps.loc[(s, name)] = 'opt infeasible'
    elif not group['feasible'].any():
        gaps.loc[(s, name)] = 'infeasible'
gaps

variant          clarke_wright       + two_opt + optimised_two_opt  \
set instance                                                         
A   A-n32-k5               7.4            5.74                5.74   
    A-n33-k5              8.32            7.56                7.56   
    A-n33-k6              5.93            5.66                5.66   
    A-n34-k5              1.67            1.67                1.67   
    A-n36-k5               2.0            0.88                0.88   
B   B-n31-k5              0.74            0.74                0.74   
    B-n34-k5              0.76            0.51                0.51   
    B-n35-k5              2.41            1.57                1.57   
    B-n38-k6              3.98             3.6                 3.6   
    B-n39-k5              2.55            2.37                2.37   
P   P-n16-k8        infeasible      infeasible          infeasible   
    P-n19-k2             11.79           11.79               11.79   
    P-n20-k2              8.33            8.33                8.33   
    P-n21-k2             11.85           11.85               11.85   
    P-n22-k8               0.0             0.0                 0.0   
X   X-n101-k25  opt infeasible  opt infeasible      opt infeasible   
    X-n106-k14            3.47            3.22                3.22   
    X-n110-k13            10.3           10.11               10.09   
    X-n115-k10            3.32            2.99                3.02   
    X-n120-k6             9.07            8.03                8.03   

variant        + relocate_and_swap + two_opt  
set instance                                  
A   A-n32-k5                            5.74  
    A-n33-k5                            6.35  
    A-n33-k6                            5.66  
    A-n34-k5                            1.67  
    A-n36-k5                            0.75  
B   B-n31-k5                            0.74  
    B-n34-k5                             0.0  
    B-n35-k5                            1.57  
    B-n38-k6                            3.48  
    B-n39-k5                            2.19  
P   P-n16-k8                      infeasible  
    P-n19-k2                           11.79  
    P-n20-k2                            8.33  
    P-n21-k2                           11.85  
    P-n22-k8                             0.0  
X   X-n101-k25                opt infeasible  
    X-n106-k14                          2.94  
    X-n110-k13                         10.11  
    X-n115-k10                          2.99  
    X-n120-k6                            7.2

In [32]:
# every variant must keep Clarke-Wright's solution feasible: this should be empty
cw[cw.groupby('instance')['feasible'].transform('any') & ~cw['feasible']]

,set,instance,variant,opt_feasible,time_ms,feasible,cost,gap_pct
